# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guided exploration of the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant schema for interoperability and reproducibility.

### Dataset Source
The dataset schema and associated data are referenced via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"\033[1m{metadata.name}\033[0m (Version {metadata.version})")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"\nDescription:\n{metadata.description}\n")

## 2. Data Overview
Explore available record sets with their `@id`s, and the corresponding fields/columns for each record set. All accesses are made via the Croissant entity `@id` attributes.

In [ ]:
# List all record sets in the dataset using their `@id`
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No explicit record sets defined in metadata. Querying any inferred or default record set IDs.")

    # Some datasets include minimal Croissant recordSet info, so try probing via records()
    # mlcroissant returns record sets as @id. We'll list up to 5 records per record set for illustration.
    all_recordset_ids = dataset.record_sets()
    if not all_recordset_ids:
        print("No record sets found in this dataset.")
    else:
        for rs_id in all_recordset_ids:
            print(f"\nRecord Set @id: {rs_id}")
            schema = dataset.schema[rs_id] if hasattr(dataset, 'schema') and rs_id in dataset.schema else None
            if schema:
                field_ids = [f["@id"] for f in schema.get('field', [])]
                print(f"  Fields: {field_ids}")
            print("  Example records:")
            # Show up to 3 records
            for i, rec in enumerate(dataset.records(record_set=rs_id)):
                print(f"    Record {i+1}: { {k:v for k,v in rec.items()} }")
                if i >= 2:
                    break
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        field_ids = [f['@id'] for f in rs.get('field',[])]
        print(f"  Fields: {field_ids}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id`.

Below, we extract all record sets and load them into a dictionary of DataFrames, using their `@id` as the key.

In [ ]:
# Get all available record set @id values
recordset_ids = dataset.record_sets()  # Returns a list of @id strings
if not recordset_ids:
    print("No record sets available to extract data from.")
else:
    dataframes = {}
    for rs_id in recordset_ids:
        print(f"\nExtracting Record Set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            print(f"Loaded {len(df)} rows; Columns (using @id):\n{list(df.columns)}")
            dataframes[rs_id] = df
        else:
            print("No records found in this record set.")
    # Preview the first DataFrame (if any record sets were found)
    if dataframes:
        sample_rs_id = list(dataframes.keys())[0]
        print(f"\nPreview of record set {sample_rs_id}:")
        display(dataframes[sample_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—such as filtering records, normalizing numeric fields, and grouping by key attributes. Entities are identified using their `@id` field.

In [ ]:
# If there is at least one dataframe, perform EDA on the first one as an example
if not dataframes:
    print('No dataframes loaded for EDA.')
else:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"\nWorking with record set: {record_set_id}")
    # Try to guess a numeric field by looking for int or float columns
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Numeric Field Selected (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.75)  # Upper quartile as example
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (top 5 rows):")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nFirst 5 normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical field
        possible_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < len(df) // 2]
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            print(f"\nGrouping by {group_field_id} (by @id):")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Use matplotlib to visualize distributions or relationships. In this section, a histogram of one numeric field (referenced by its @id) is displayed, along with a simple boxplot by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print('No data available for visualization.')
else:
    # Work with the previously selected record set/dataframe
    df = dataframes[record_set_id]
    if not numeric_fields:
        print("No numeric field available for visualization.")
    else:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f"Histogram of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        # Try to visualize with a categorical group if available
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            plt.figure(figsize=(10,5))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library, referencing all entities and fields by their `@id`. This guided workflow provides a reproducible foundation for deeper domain modeling, custom feature selection, and downstream statistical or machine learning analysis. Adjust the filtering and grouping operations depending on your research question and the specific schema of the dataset.

For more information about the dataset and its metadata, consult the Croissant schema:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`
